In [16]:
import pandas as pd
import numpy as np 
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
planets = pd.read_csv("./planetary_systems.csv", skiprows=1)
print(planets.shape) 
planets.dropna(inplace=True)
planets.reset_index(inplace=True)
planets.drop("index", axis=1, inplace=True) 

print(planets.columns) 
print(planets.shape) 

(5613, 29)
Index(['pl_controv_flag', 'pl_name', 'hostname', 'sy_snum', 'sy_pnum',
       'discoverymethod', 'disc_year', 'disc_facility', 'pl_orbper',
       'pl_orbsmax', 'pl_rade', 'pl_bmasse', 'pl_orbeccen', 'pl_eqt',
       'st_spectype', 'st_teff', 'st_rad', 'st_mass', 'st_met', 'st_metratio',
       'st_logg', 'rastr', 'ra', 'decstr', 'dec', 'sy_dist', 'sy_vmag',
       'sy_kmag', 'sy_gaiamag'],
      dtype='object')
(861, 29)


In [18]:
planets.head() 

,pl_controv_flag,pl_name,hostname,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,pl_orbper,pl_orbsmax,...,st_metratio,st_logg,rastr,ra,decstr,dec,sy_dist,sy_vmag,sy_kmag,sy_gaiamag
0,0.0,51 Eri b,51 Eri,3,1,Imaging,2015,Gemini Observatory,11688,13.2,...,[M/H],4.31,04h37m36.18s,69.4007424,-02d28m25.77s,-2.4738245,29.7575,5.21149,4.537,5.15806
1,0.0,55 Cnc b,55 Cnc,2,5,Radial Velocity,1996,Lick Observatory,14.6516,0.1134,...,[Fe/H],4.43,08h52m35.24s,133.1468373,+28d19m47.34s,28.3298154,12.5855,5.95084,4.015,5.72973
2,0.0,55 Cnc e,55 Cnc,2,5,Radial Velocity,2004,McDonald Observatory,0.7365474,0.01544,...,[Fe/H],4.43,08h52m35.24s,133.1468373,+28d19m47.34s,28.3298154,12.5855,5.95084,4.015,5.72973
3,0.0,AF Lep b,AF Lep,1,1,Imaging,2023,Paranal Observatory,8030,8.4,...,[Fe/H],4.4,05h27m04.78s,81.7699216,-11d54m04.23s,-11.9011752,26.8564,6.332,4.926,6.18644
4,0.0,AU Mic b,AU Mic,1,3,Transit,2020,Transiting Exoplanet Survey Satellite (TESS),8.4629991,0.0645,...,[M/H],4.4,20h45m09.87s,311.2911369,-31d20m32.82s,-31.34245,9.7221,8.81,4.529,7.84038


In [ ]:
#goal is to make a function where you can define the planets and top n similar planets

def similar_planets(list_of_planets, n_similar_planets):
    scores = [] 
    

In [19]:
result = planets.loc[:, ['sy_snum', 'sy_pnum', 'pl_orbper', 'pl_orbsmax']]
result

,sy_snum,sy_pnum,pl_orbper,pl_orbsmax
0,3,1,11688,13.2
1,2,5,14.6516,0.1134
2,2,5,0.7365474,0.01544
3,1,1,8030,8.4
4,1,3,8.4629991,0.0645
...,...,...,...,...
856,1,3,4.65626,0.02851
857,1,2,8617.50952,10.018
858,1,3,6.26791,0.069
859,1,4,39.8438,0.2245


In [20]:
result.head(2) 

,sy_snum,sy_pnum,pl_orbper,pl_orbsmax
0,3,1,11688,13.2
1,2,5,14.6516,0.1134


In [30]:
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances

In [31]:
hey = result.iloc[0:1,]  
hey2 = np.array(hey) 
print(hey2) 
second = result.iloc[1:2,] 
second2 = np.array(second)
print(second2) 

#using cosine similary for proof of concept
similarity = cosine_similarity(hey2,second2) 
print(similarity) 

[['3' '1' '11688' '13.2']]
[['2' '5' '14.6516' '0.1134']]
[[0.93865179]]


In [94]:
#goal is to make a function where you can define the planets and top n similar planets


from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import numpy as np

def find_similar_planets(target_planet, n_similar_planets):
    if target_planet not in planets['pl_name'].values:
        print(f"Planet '{target_planet}' not found.")
        return

    feature_cols = ['sy_snum', 'sy_pnum', 'pl_orbper', 'pl_orbsmax']
    feature_df = planets[['pl_name'] + feature_cols].dropna().reset_index(drop=True)

    scaler = StandardScaler()
    feature_df[feature_cols] = scaler.fit_transform(feature_df[feature_cols])

    target_row = feature_df[feature_df['pl_name'] == target_planet]
    target_vector = np.array(target_row[feature_cols])
    
    other_rows = feature_df[feature_df['pl_name'] != target_planet].reset_index(drop=True)
    other_vectors = np.array(other_rows[feature_cols])

    print("Target vector:\n", target_vector)
    print("Sample other vectors:\n", other_vectors[:3])

    similarities = cosine_similarity(target_vector, other_vectors)[0]
    results = list(zip(other_rows['pl_name'], similarities))
    top_n = sorted(results, key=lambda x: x[1], reverse=True)[:n_similar_planets]

    print(f"\nTop {n_similar_planets} planets similar to '{target_planet}':")
    for planet, score in top_n:
        print(f"  {planet}: similarity = {score:.3f}")


     






In [95]:
find_similar_planets('51 Eri b', 10)  

Target vector:
 [[ 4.00325683 -0.72860531  2.55408577  5.42130867]]
Sample other vectors:
 [[ 1.79234975  2.265806   -0.09419859 -0.11053592]
 [ 1.79234975  2.265806   -0.09735544 -0.15194465]
 [-0.41855732 -0.72860531  1.72421041  3.39229777]]

Top 10 planets similar to '51 Eri b':
  HD 190360 b: similarity = 0.959
  HIP 21152 b: similarity = 0.917
  HR 5183 b: similarity = 0.897
  bet Pic b: similarity = 0.772
  AF Lep b: similarity = 0.767
  HR 8799 d: similarity = 0.766
  HR 8799 c: similarity = 0.762
  HR 8799 e: similarity = 0.756
  PDS 70 b: similarity = 0.735
  GJ 414 A c: similarity = 0.725


In [75]:
#goal is to get a combination of all of the numbers with
# a) no repeats 
# b) no number matched with each other 

import random 

list1 = [1,2,3,4]
list2 = {} 

for i in range(len(list1)-1):
    list2[list1[i]] = [] 
    for j in range(i + 1, len(list1)):
        #list2[list1[i]].append([i,j])  
        list2[list1[i]].append(random.randint(5, 25))  
        

print(list2)   

#getting the top n scores 
# list3 = []
# for x, y in list2.items():
#     list3.append(y) 
# list3 

for x, y in list2.items():
    print(y) 

print(list2.items())

flattened = [num for y in list2.values() for num in y] 
print(flattened) 
max_value = sorted(flattened, reverse = True)[:5]   
print(max_value) 

#max_value = max(list2.values())  
#print(f'Max Value is {max_value}') 


{1: [7, 11, 18], 2: [19, 24], 3: [5]}
[7, 11, 18]
[19, 24]
[5]
dict_items([(1, [7, 11, 18]), (2, [19, 24]), (3, [5])])
[7, 11, 18, 19, 24, 5]
[24, 19, 18, 11, 7]


In [83]:
listi = ['51 Eri b', '55 Cnc b']  
filtered = planets[planets['pl_name'].isin(listi)]
filtered 
print(planets.head(4)) 

   pl_controv_flag   pl_name hostname sy_snum sy_pnum  discoverymethod  \
0              0.0  51 Eri b   51 Eri       3       1          Imaging   
1              0.0  55 Cnc b   55 Cnc       2       5  Radial Velocity   
2              0.0  55 Cnc e   55 Cnc       2       5  Radial Velocity   
3              0.0  AF Lep b   AF Lep       1       1          Imaging   

  disc_year         disc_facility  pl_orbper pl_orbsmax  ... st_metratio  \
0      2015    Gemini Observatory      11688       13.2  ...       [M/H]   
1      1996      Lick Observatory    14.6516     0.1134  ...      [Fe/H]   
2      2004  McDonald Observatory  0.7365474    0.01544  ...      [Fe/H]   
3      2023   Paranal Observatory       8030        8.4  ...      [Fe/H]   

  st_logg         rastr           ra         decstr          dec  sy_dist  \
0    4.31  04h37m36.18s   69.4007424  -02d28m25.77s   -2.4738245  29.7575   
1    4.43  08h52m35.24s  133.1468373  +28d19m47.34s   28.3298154  12.5855   
2    4.43  08h52m

In [11]:
def cosine_calculations(dataframe):
    exoplanet_similarity={}
    for index in range(len(dataframe)):
        level_of_similarity = cosine_similarity(dataframe.loc[index:index], [[3,2,5,4]])
        exoplanet_similarity[index] = level_of_similarity[0][0] 
    
    sorted_list = sorted(exoplanet_similarity.items(), key=lambda item: item[1], reverse=True)
    sorted_dict_by_similarity = dict(sorted_list) 

    top_five = dict(list(sorted_dict_by_similarity.items())[:5])
    keys_list = list(top_five.keys()) 
    planet_names = planets.loc[keys_list, "pl_name"].tolist()  

    return planet_names 


In [14]:
cosine_calculations(result) 
# hey

['KELT-18 b', 'MASCARA-4 b', 'HATS-30 b', 'WASP-100 b', 'WASP-123 b']